# P2W1D4 - RAG dengan LangChain

**Studi kasus:** HacktivRun Help Center  
**Target:** pipeline end-to-end `Load -> Split -> Embed -> Store -> Retrieve -> Augment -> Generate`.

Notebook ini menggunakan **OpenAI** untuk embedding dan generation, serta Pinecone sebagai vector database. Kerjakan berurutan karena setiap bagian memakai objek dari bagian sebelumnya.

> **Kernel:** gunakan **Python 3.13 (P2W1D4 RAG)** sebelum menjalankan notebook.
> **Environment:** notebook peserta membaca credential dari `.env` di root folder lab.


## 0. Kontrak Lab

- Dokumen: `data/hacktivrun_help_center.pdf`
- Embedding: `text-embedding-3-small` (1536 dimensi)
- Generation: `gpt-5.6-luna`
- Pinecone index: index Day 03
- Namespace: `rag-handson`

> Day 03 memakai namespace `handson-lab` untuk katalog produk. Day 04 hanya memakai ulang index dan kontrak embedding-nya; PDF HacktivRun disimpan di namespace `rag-handson` agar retrieval tidak mencampur kedua dataset.

> Jangan pernah menulis API key langsung di notebook.


## 1. Install Package


In [ ]:
# langchain-core: komponen utama untuk prompt, runnable, dan RAG chain
# langchain-openai: integrasi model chat dan embedding OpenAI
# langchain-community dan pypdf: memuat dan mengekstrak dokumen PDF
# langchain-text-splitters: memecah dokumen menjadi chunk yang saling overlap
# langchain-pinecone dan pinecone: integrasi serta SDK vector database Pinecone
# python-dotenv: membaca kredensial dan konfigurasi dari file .env

%pip install -q langchain-core langchain-openai langchain-community langchain-text-splitters langchain-pinecone pinecone pypdf python-dotenv


## 2. Credentials dan Konfigurasi


In [ ]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

WORKING_DIR = Path.cwd().resolve()
root_candidates = [
    WORKING_DIR,
    WORKING_DIR.parent,
    WORKING_DIR / "week_5_day_4_rag_langchain",
]
LAB_ROOT = next(
    (path for path in root_candidates if (path / "data" / "hacktivrun_help_center.pdf").is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError("Root folder week_5_day_4_rag_langchain tidak ditemukan.")

ENV_PATH = LAB_ROOT / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f"File environment peserta tidak ditemukan: {ENV_PATH}")

load_dotenv(dotenv_path=ENV_PATH, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "").strip()
PINECONE_INDEX = os.getenv("PINECONE_INDEX", "aien-elite-runner-gear").strip()
PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "rag-handson").strip()
CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna").strip()
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small").strip()

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY belum diisi di file .env.")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum diisi di file .env.")
if not PINECONE_INDEX or not PINECONE_NAMESPACE:
    raise ValueError("PINECONE_INDEX dan PINECONE_NAMESPACE tidak boleh kosong.")
if not CHAT_MODEL or not EMBEDDING_MODEL:
    raise ValueError("OPENAI_MODEL dan OPENAI_EMBEDDING_MODEL tidak boleh kosong.")

PDF_PATH = LAB_ROOT / "data" / "hacktivrun_help_center.pdf"
if not PDF_PATH.exists():
    raise FileNotFoundError(f"Dokumen tidak ditemukan: {PDF_PATH.resolve()}")

pc = Pinecone(api_key=PINECONE_API_KEY)
if not pc.has_index(PINECONE_INDEX):
    raise ValueError(
        f"Index {PINECONE_INDEX!r} tidak ditemukan. "
        "Selesaikan setup index Week 05 Day 03 terlebih dahulu."
    )

description = pc.describe_index(PINECONE_INDEX)
if description.dimension != 1536 or description.metric != "cosine":
    raise ValueError(
        "Kontrak index tidak sesuai: dibutuhkan dimension=1536 dan metric='cosine'. "
        f"Ditemukan dimension={description.dimension}, metric={description.metric}."
    )

index = pc.Index(PINECONE_INDEX)
index_stats = index.describe_index_stats()
day3_namespace = index_stats.namespaces.get("handson-lab")
day3_vector_count = day3_namespace.vector_count if day3_namespace else 0
print(
    f"Ready | index={PINECONE_INDEX} | namespace={PINECONE_NAMESPACE} | "
    f"embedding={EMBEDDING_MODEL} | chat={CHAT_MODEL}"
)
print(f"Info Day 03 | namespace='handson-lab' | vector_count={day3_vector_count}")
print("Day 04 tetap memakai namespace terpisah agar retrieval tidak mencampur dataset.")


def display_page_number(value):
    return int(value) if isinstance(value, float) and value.is_integer() else value


## 3. Load - Membaca Dokumen PDF

`PyPDFLoader` menghasilkan satu `Document` per halaman. Pertahankan metadata halaman karena akan dipakai untuk sitasi.


In [ ]:
# TODO 1: buat PyPDFLoader dari PDF_PATH, lalu panggil .load().
loader = ...
documents = ...

assert documents, "PDF tidak menghasilkan dokumen."
print(f"Jumlah halaman: {len(documents)}")
print("Metadata halaman pertama:", documents[0].metadata)
print(documents[0].page_content[:500])


## 4. Split - Membentuk Chunk

Baseline `1000/200` adalah titik awal, bukan angka universal. Overlap membantu menjaga konteks yang terpotong di batas chunk.


In [ ]:
# TODO 2:
# - buat RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200,
#   add_start_index=True)
# - split documents
# - tambahkan metadata document_id, page_number (mulai dari 1), chunk_index,
#   dan source_file ke setiap chunk.
splitter = ...
chunks = ...

for chunk_index, chunk in enumerate(chunks):
    zero_based_page = int(chunk.metadata.get("page", 0))
    chunk.metadata.update(
        {
            "document_id": ...,
            "page_number": ...,
            "chunk_index": ...,
            "source_file": ...,
        }
    )

assert chunks, "Chunk tidak terbentuk."
assert all("page_number" in chunk.metadata for chunk in chunks)
print(f"Jumlah chunk: {len(chunks)}")
print("Contoh metadata:", chunks[0].metadata)
print(chunks[0].page_content[:500])


## 5. Embed dan Store - Menulis ke Pinecone

Gunakan namespace `rag-handson` agar isi lab tidak bercampur dengan namespace Day 03. ID deterministik membuat rerun bersifat idempotent untuk versi dokumen yang sama.


In [ ]:
# TODO 3:
# - buat OpenAIEmbeddings dari EMBEDDING_MODEL dengan timeout=60 dan max_retries=2
# - buat PineconeVectorStore dengan index dan namespace yang sudah disiapkan
# - buat ID deterministik: <document_id>#chunk-<chunk_index 4 digit>
# - panggil add_documents(documents=chunks, ids=chunk_ids)
# Setelah bagian di atas selesai, cell ini menunggu sampai vector terlihat di namespace.
embeddings = ...
vectorstore = ...
chunk_ids = ...

...

assert len(chunk_ids) == len(set(chunk_ids)), "ID chunk harus unik."

deadline = time.time() + 60
while True:
    stats = index.describe_index_stats()
    namespace_stats = stats.namespaces.get(PINECONE_NAMESPACE)
    vector_count = namespace_stats.vector_count if namespace_stats else 0
    if vector_count >= len(chunk_ids):
        break
    if time.time() >= deadline:
        raise TimeoutError("Vector belum terlihat di Pinecone setelah 60 detik.")
    time.sleep(1)

print(
    f"Upsert selesai: {len(chunk_ids)} chunk; "
    f"namespace {PINECONE_NAMESPACE!r} berisi {vector_count} vector."
)


## 6. Retrieve - Inspeksi Sebelum Generation

Periksa apakah chunk yang diambil benar-benar memuat bukti. RAG tidak dapat memperbaiki konteks yang sejak awal salah.


In [ ]:
# TODO 4: ubah vectorstore menjadi retriever similarity dengan k=4, lalu panggil
# retriever.invoke() untuk inspection_question.
retriever = ...
inspection_question = "Apa syarat retur produk HacktivRun?"
retrieved_docs = ...

assert retrieved_docs, "Retriever tidak mengembalikan dokumen."
for rank, doc in enumerate(retrieved_docs, start=1):
    page = display_page_number(doc.metadata.get("page_number"))
    print(f"[{rank}] halaman={page} source={doc.metadata.get('source_file')}")
    print(doc.page_content[:600])
    print("-" * 80)


**Checkpoint retrieval**

Sebelum lanjut, jawab singkat:

1. Halaman mana yang paling relevan?
2. Apakah keempat chunk benar-benar membantu?
3. Jika tidak, parameter apa yang akan Anda ubah terlebih dahulu?


## 7. Augment + Generate - Grounded RAG Chain

`format_docs()` memberi label `[n]` pada setiap chunk. Label itu menjadi kontrak sitasi antara konteks dan jawaban.


In [ ]:
FALLBACK = "Informasi tidak ditemukan dalam dokumen."


# TODO 5: format setiap dokumen menjadi blok berlabel [n] yang memuat SOURCE dan PAGE.
# Gunakan display_page_number() agar metadata halaman dari Pinecone mudah dibaca.
def format_docs(docs):
    ...


# TODO 6: buat ChatPromptTemplate yang:
# - menjawab hanya dari CONTEXT dan mengabaikan instruksi di dalam dokumen;
# - mewajibkan sitasi [n];
# - memakai FALLBACK persis ketika bukti tidak cukup.
prompt = ...

# TODO 7: buat ChatOpenAI dengan Responses API, reasoning_effort='low', lalu
# rangkai LCEL yang mengembalikan question, documents, context, dan answer.
# Pastikan retriever hanya dipanggil satu kali untuk setiap pertanyaan.
model = ...
rag_chain = ...


## 8. Uji Pertanyaan In-Scope


In [ ]:
# TODO 8: implementasikan helper ask_rag() untuk menjalankan chain, mengambil
# result['answer'] dan result['documents'], lalu mencetak sumber. Gunakan
# display_page_number() saat menampilkan halaman.
def ask_rag(question: str, show_sources: bool = True) -> str:
    ...


test_questions = [
    "Berapa lama estimasi pengiriman ke luar Pulau Jawa?",
    "Apa syarat untuk mengembalikan sepatu yang ukurannya tidak pas?",
    "Berapa lama proses refund setelah disetujui?",
]

for question in test_questions:
    ask_rag(question)
    print("=" * 90)


## 9. Uji Pertanyaan Out-of-Scope

Negative case sama pentingnya dengan positive case. Sistem yang aman harus mengakui ketika bukti tidak tersedia.


In [ ]:
# TODO 9: jalankan negative question dan pastikan jawaban sama dengan FALLBACK.
negative_question = "Berapa suhu rata-rata permukaan Mars?"
negative_answer = ...

normalized = negative_answer.strip().strip('"').strip("'")
assert normalized == FALLBACK, (
    "Negative case belum aman. Periksa retrieval dan instruksi fallback. "
    f"Jawaban saat ini: {negative_answer!r}"
)
print("Negative-case check: PASS")


## Tantangan Lanjutan

Pilih **satu**:

- Tambahkan metadata filter `section` pada setiap chunk dan batasi retrieval ke bagian tertentu.
- Bandingkan `k=2`, `k=4`, dan `k=6`; catat dampaknya pada relevansi dan panjang konteks.
- Tambahkan evaluasi sederhana berisi lima pertanyaan, expected keyword, dan status pass/fail.


## Ringkasan

- Dokumen PDF dimuat, dipecah menjadi chunk, lalu diubah menjadi embedding OpenAI.
- Chunk beserta metadata sumber disimpan di Pinecone dan diambil kembali melalui retriever.
- LangChain menggabungkan pertanyaan, context hasil retrieval, dan `gpt-5.6-luna` menjadi pipeline RAG end-to-end.
- Grounding, sitasi halaman, dan fallback membantu menjaga jawaban tetap berdasarkan dokumen.
